# The t-procedure
Student's t-distribution against the standard normal, t critical values, the coverage of "z with s" and "t with s", and a t-interval for the mean Titanic fare.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats

## 1. Student's t against the standard normal

In [ ]:
x = np.linspace(-5, 5, 400)
fig = go.Figure(go.Scatter(x=x, y=stats.norm.pdf(x), name="standard normal"))
for df in [1, 4, 29]:
    fig.add_scatter(x=x, y=stats.t.pdf(x, df), name=f"t, df = {df}", line_dash="dash")
fig.update_layout(template="simple_white")
fig.show()
# tail area beyond 3: much larger for small df
for df in [1, 4, 29, 1000]:
    print(f"df {df:4d}: P(T > 3) = {stats.t.sf(3, df):.4f}")
print(f"normal:  P(Z > 3) = {stats.norm.sf(3):.4f}")

## 2. Critical values: two-sided 95% (area 0.975 to the left)

In [ ]:
for df in [1, 4, 9, 29, 30, 40, 60, 100, 1000]:
    print(f"df {df:4d}: t = {stats.t.ppf(0.975, df):.3f}   (one-sided 95%, area 0.95: {stats.t.ppf(0.95, df):.3f})")
print("z =", round(stats.norm.ppf(0.975), 3))

## 3. Coverage: z with s against t with s, population N(50, 15^2)

In [ ]:
def coverage(n, kind, sims=100_000, seed=42):
    xs = np.random.default_rng(seed).normal(50, 15, (sims, n))
    m, s = xs.mean(axis=1), xs.std(axis=1, ddof=1)      # sample standard deviation (n - 1)
    crit = stats.norm.ppf(0.975) if kind == "z" else stats.t.ppf(0.975, n - 1)
    e = crit * s / np.sqrt(n)
    return np.mean((m - e <= 50) & (50 <= m + e))

for n in [5, 10, 30, 100]:
    print(f"n = {n:3d}: z with s {coverage(n, 'z'):.1%}   t with s {coverage(n, 't'):.1%}")

## 4. Titanic: a 95% t-interval for the mean fare from one sample of 30

In [ ]:
df = pd.concat([pd.read_csv("data/titanic_train.csv").drop(columns="Survived"),
                pd.read_csv("data/titanic_test.csv")])
fares = df["Fare"].dropna().to_numpy()       # the population (only to check the answer)
rng = np.random.default_rng(42)
sample = rng.choice(fares, 30, replace=False)
x_bar, s, n = sample.mean(), sample.std(ddof=1), len(sample)
t = stats.t.ppf(0.975, n - 1)                # df = 29
print(f"x_bar {x_bar:.2f}, s {s:.2f}, t {t:.3f}")
print(f"95% CI: {x_bar - t * s / np.sqrt(n):.2f} to {x_bar + t * s / np.sqrt(n):.2f}")
print(stats.t.interval(0.95, n - 1, loc=x_bar, scale=s / np.sqrt(n)))
t50 = stats.t.ppf(0.75, n - 1)
print(f"50% CI (t = {t50:.3f}): {x_bar - t50 * s / np.sqrt(n):.2f} to {x_bar + t50 * s / np.sqrt(n):.2f}")
print("true mean fare:", round(fares.mean(), 2))

## 5. Ten samples of 30: average them, or pool them

In [ ]:
rng = np.random.default_rng(42)
ten = np.array([rng.choice(fares, 30, replace=False) for _ in range(10)])
avg_mean, avg_s = ten.mean(axis=1).mean(), ten.std(axis=1, ddof=1).mean()
print(f"average of the means {avg_mean:.2f}, average of the s {avg_s:.2f}")
print(f"interval using n = 30: {avg_mean - 2.045 * avg_s / np.sqrt(30):.2f} to {avg_mean + 2.045 * avg_s / np.sqrt(30):.2f}")
pooled = ten.ravel()                          # 300 fares in one sample
print("pooled, n = 300:", stats.t.interval(0.95, 299, loc=pooled.mean(), scale=pooled.std(ddof=1) / np.sqrt(300)))

## 6. How often does the t-interval catch the mean of these skewed fares?

In [ ]:
mu = fares.mean()
for n in [30, 100]:
    hit = 0
    for r in range(4000):
        y = np.random.default_rng(9000 + r).choice(fares, n, replace=False)
        e = stats.t.ppf(0.975, n - 1) * y.std(ddof=1) / np.sqrt(n)
        hit += abs(y.mean() - mu) <= e
    print(f"n = {n}: {hit / 4000:.1%}")